Crearemos un RPG, en donde tendremos un personaje que debe ganar x niveles, mejorará sus stats al vencer cada nivel (1 nivel es una batalla) y obtendrá mejoras.

Definiremos las siguientes clases:


*   Jugador
*   Humano/Enemigo
*   Juego

Los jugadores tendrán un nivel de salud, tendrán 2 acciones o atacar o defenderse, tendrán un "nivel de experiencia", tendrán una cantidad de experiencia, y al vencer cada nivel aumentará su experiencia y ataque en 1.



In [ ]:
from google.colab import output
import random

In [ ]:
class Jugador:
  def __init__(self, name="Jugador", hp=100, xp=0, atk1_dmg=5, suffix=""):
    self.name = name
    self.hp = hp
    self.xp = xp
    self.atk1_dmg = atk1_dmg
    self.block = False
    self.dead = False
    self.name += suffix

  def attack(self, target):
    dmg_done = target.take_damage(self.atk1_dmg)
    return dmg_done

  def take_damage(self, dmg):
    if self.block == False:
      self.hp -= dmg
      return dmg
    else:
      return 0

class Enemigo(Jugador):
  def __init__(self, *args, name="Enemigo", **kwargs):
    kwargs["name"] = name
    super().__init__(*args, **kwargs)

  def get_action(self):
    return random.choice([1, 2])

  def get_target(self, targets):
    return targets[0]

class Slime(Enemigo):
  def __init__(self, suffix=""):
    super().__init__(name="Slime", hp=10, atk1_dmg=5, suffix=suffix)

class Bruja(Enemigo):
  def __init__(self, suffix=""):
    super().__init__(name="Bruja", hp=30, atk1_dmg=10, suffix=suffix)

class Esqueleto(Enemigo):
  def __init__(self, suffix=""):
    super().__init__(name="Esqueleto", hp=20, atk1_dmg=5, suffix=suffix)

class Zombie(Enemigo):
  def __init__(self, suffix=""):
    super().__init__(name="Zombie", hp=20, atk1_dmg=10, suffix=suffix)

class Ogro(Enemigo):
  def __init__(self, suffix=""):
    super().__init__(name="Ogro", hp=100, atk1_dmg=20, suffix=suffix)

class Humano(Jugador):
  def __init__(self, *args, name="Humano", **kwargs):
    kwargs["name"] = name
    super().__init__(*args, **kwargs)

  def get_action(self):
    while True:
      print("Elige una acción:")
      print("1. Atacar")
      print("2. Defenderse")
      try:
        accion = int(input())
        if accion not in [1, 2]:
          raise ValueError

        break
      except ValueError:
        print("Acción incorrecta")
        continue
    return accion

  def get_target(self, targets):
    while True:
      print("Elige un enemigo:")
      dead_ids = [i+1 for i, enemigo in enumerate(targets) if enemigo.dead]
      for i, enemigo in enumerate(targets):
        if not enemigo.dead:
          name_to_display = f"{enemigo.name} HP = {enemigo.hp}"
        else:
          name_to_display = "Eliminado 💀💀💀"
        print(f"{i+1}. {name_to_display}")
      try:
        id_enemigo = int(input())
        if id_enemigo not in [i+1 for i in range(len(targets))] or id_enemigo in dead_ids:
          raise ValueError
        break
      except ValueError:
        print("Selección incorrecta")
        continue
    return targets[id_enemigo-1]

class Nivel:
  def __init__(self, name="Nivel", enemies=[], rewards=[]):
    self.name = name
    self.enemies = enemies
    self.rewards = rewards

class Premio:
  # Enum de premios
  AUMENTAR_HP = 1
  AUMENTAR_ATK1 = 2

class Accion:
  # Enum de acciones
  ATACAR = 1
  DEFENDERSE = 2

class Juego:

  # Nivel default
  level_pkg_1 = [
    Nivel("Nivel 1", [Slime()], rewards=[Premio.AUMENTAR_HP, Premio.AUMENTAR_ATK1]),
    Nivel("Nivel 2", [Slime(suffix=" 1"), Slime(suffix=" 2")]),
    Nivel("Nivel 3", [Zombie(), Slime()], rewards=[Premio.AUMENTAR_HP]),
    Nivel("Nivel 4", [Zombie(), Esqueleto(), Bruja()], rewards=[Premio.AUMENTAR_HP, Premio.AUMENTAR_ATK1]),
    Nivel("Nivel 5", [Ogro()])
  ]

  def __init__(self, player_name="Humano", level_pkg=None):
    self.jugador = Humano(name=player_name)
    self.nivel = 0
    self.turno = 0
    self.curr_level = None
    self.curr_player = None
    if level_pkg is None: self.level_pkg = Juego.level_pkg_1
    else: self.level_pkg = level_pkg

  @staticmethod
  def create_random_level_package(n_levels=5):
    pkg = []
    for i in range(n_levels):
      n_enemies = random.randint(1, 3)
      n_rewards = random.randint(0, 2)
      enemies = [random.choice([Slime, Slime, Esqueleto, Zombie, Bruja, Ogro]) for i in range(n_enemies)]
      rewards = [random.choice([Premio.AUMENTAR_HP, Premio.AUMENTAR_ATK1]) for i in range(n_rewards)]
      nivel = Nivel(f"Nivel {i+1}", enemies=[enemy() for enemy in enemies], rewards=rewards)
      pkg.append(nivel)
    return pkg

  def start_game(self):
    self.__main_loop()

  def __select_reward(self):
    if self.curr_level.rewards == []:
      return None
    while True:
      print("Elige un premio:")
      for i, premio in enumerate(self.curr_level.rewards):
        if premio == Premio.AUMENTAR_HP:
          print(f"{i+1}. Aumentar HP")
        elif premio == Premio.AUMENTAR_ATK1:
          print(f"{i+1}. Aumentar ATK1")
      try:
        seleccion = int(input())
        if seleccion not in [i+1 for i, _ in enumerate(self.curr_level.rewards)]:
          raise ValueError
        break
      except ValueError:
        print("Premio incorrecto")
        continue
    return seleccion

  def __set_curr_player(self):
    n_players = 1 + len(self.curr_level.enemies)
    turno_truncado = self.turno%n_players
    if turno_truncado == 0:
      self.curr_player = self.jugador
    else:
      self.curr_player = self.curr_level.enemies[self.turno%n_players-1]

  def __show_game_stats(self):

    input()
    output.clear()

    print(f"{self.curr_level.name}, Turno {self.turno}")
    print("Datos del jugador:")
    print(f"HP = {self.jugador.hp}, XP = {self.jugador.xp}, ATK1 = {self.jugador.atk1_dmg}")
    print("Datos de los enemigos:")
    for enemigo in self.curr_level.enemies:
      if not enemigo.dead:
        print(f"{enemigo.name} HP = {enemigo.hp}")
    print("-"*27)

  def __execute_action(self, accion):
    if accion == Accion.ATACAR:
      if isinstance(self.curr_player, Humano): targets = self.curr_level.enemies
      else: targets = [self.jugador]
      target = self.curr_player.get_target(targets)
      atk_dmg = self.curr_player.attack(target)
      output.clear()
      print(f"{self.curr_player.name} decidió atacar a {target.name} con daño de -{atk_dmg}.")
    elif accion == Accion.DEFENDERSE:
      self.curr_player.block = True
      print(f"{self.curr_player.name} decidió defenderse.")

  def __execute_reward(self, seleccion):
    if seleccion == Premio.AUMENTAR_HP:
        delta_hp = random.randint(5, 20)
        self.jugador.hp += delta_hp
        print(f"Tu HP aumentó en {delta_hp}!")
    elif seleccion == Premio.AUMENTAR_ATK1:
        delta_atk1 = random.randint(3, 6)
        self.jugador.atk1_dmg += delta_atk1
        print(f"Tu ATK1 aumentó en {delta_atk1}!")

  def __show_new_dead_enemies(self):
    for enemigo in self.curr_level.enemies:
      if enemigo.hp <= 0 and not enemigo.dead:
        enemigo.dead = True
        print(f"El enemigo {enemigo.name} ha muerto.")

  def __main_loop(self):

    while True:

      self.curr_level = self.level_pkg[self.nivel]

      self.__set_curr_player()

      # Saltar jugador si está muerto
      if self.curr_player.dead:
        self.turno += 1
        continue

      # Clean defense trigger
      if self.curr_player.block:
        self.curr_player.block = False

      self.__show_game_stats()

      accion = self.curr_player.get_action()
      output.clear()
      self.__execute_action(accion)

      # Evaluar estado del juego

      self.__show_new_dead_enemies()

      all_enemies_dead = all([enemigo.dead for enemigo in self.curr_level.enemies])
      human_dead = self.jugador.dead

      if human_dead:
        print("Has perdido el juego.")
        break

      if all_enemies_dead:
        print("Has ganado el nivel.")
        self.nivel += 1
        self.turno = 0
        seleccion = self.__select_reward()
        if seleccion is not None: self.__execute_reward(seleccion)
        if self.nivel == len(self.level_pkg):
          print("Has ganado el juego.")
          break
        continue

      self.turno += 1

In [ ]:
custom_pkg = Juego.create_random_level_package(n_levels=3)

In [ ]:
juego = Juego(level_pkg=custom_pkg)

In [ ]:
juego.start_game()

Humano decidió atacar a Bruja con daño de -5.
El enemigo Bruja ha muerto.
Has ganado el nivel.
Elige un premio:
1. Aumentar HP
1
Tu HP aumentó en 12!
Has ganado el juego.
